# Interpretación de los Intervalos de Confianza

Este notebook tiene como objetivo ilustrar de forma gráfica y matemática el significado real de un **nivel de confianza**.

Mucha gente interpreta erróneamente que un intervalo de confianza del 95% significa que hay un 95% de probabilidad de que el parámetro poblacional esté dentro de *ese* intervalo específico. En realidad, el nivel de confianza hace referencia al método: si repetimos el experimento muchas veces, aproximadamente el 95% de los intervalos calculados contendrán el verdadero parámetro poblacional.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats

# Configuración de semilla para reproducibilidad
np.random.seed(42)

## 1. Configuración de la Simulación

Definiremos una población normal con una media ($\mu$) y desviación estándar ($\sigma$) conocidas. Luego, realizaremos 100 corridas (simulaciones). En cada corrida, tomaremos una muestra de tamaño $n = 30$ y calcularemos su intervalo de confianza del 95%.

In [ ]:
# Parámetros de la población
mu_poblacion = 100
sigma_poblacion = 15

# Parámetros de la simulación
tamanyo_muestra = 30
n_corridas = 100
nivel_confianza = 0.95

# Listas para almacenar los resultados
intervalos = []
medias_muestrales = []
contiene_parametro = []

# Valor crítico Z para un intervalo de confianza del 95%
z_critico = stats.norm.ppf(1 - (1 - nivel_confianza) / 2)

for i in range(n_corridas):
    # 1. Extraer una muestra aleatoria de la población
    muestra = np.random.normal(loc=mu_poblacion, scale=sigma_poblacion, size=tamanyo_muestra)

    # 2. Calcular estadísticas de la muestra
    media_m = np.mean(muestra)
    error_estandar = sigma_poblacion / np.sqrt(tamanyo_muestra) # Usamos sigma conocida para simplicidad

    # 3. Calcular límites del intervalo
    limite_inferior = media_m - z_critico * error_estandar
    limite_superior = media_m + z_critico * error_estandar

    # 4. Verificar si el intervalo contiene a la media poblacional real (mu = 100)
    exito = limite_inferior <= mu_poblacion <= limite_superior

    # Guardar resultados
    medias_muestrales.append(media_m)
    intervalos.append((limite_inferior, limite_superior))
    contiene_parametro.append(exito)

# Calcular la proporción de intervalos que contienen el parámetro
proporcion_exito = np.mean(contiene_parametro)
print(fr"Proporción de intervalos que contienen a la media poblacional (μ={mu_poblacion}): {proporcion_exito:.2f}")

## 2. Visualización Gráfica de las 100 Corridas

En el siguiente gráfico, representaremos cada uno de los 100 intervalos calculados como una línea horizontal. La línea vertical discontinua representa el valor real de la media poblacional ($\mu = 100$).

* Los intervalos en **verde** son aquellos que lograron capturar o incluir a la media poblacional.
* Los intervalos en **rojo** son aquellos que fallaron en incluirla.

In [ ]:
plt.figure(figsize=(12, 8))

# Dibujar la línea de la verdadera media poblacional usando el carácter unicode de mu directamente
plt.axvline(x=mu_poblacion, color='black', linestyle='--', linewidth=2, label=f'Media Poblacional Real (μ = {mu_poblacion})')

# Graficar cada intervalo
for i in range(n_corridas):
    lim_inf, lim_sup = intervalos[i]
    color = 'green' if contiene_parametro[i] else 'red'

    # Graficar el intervalo de confianza como una línea horizontal
    plt.plot([lim_inf, lim_sup], [i, i], color=color, alpha=0.7, linewidth=1.5)
    # Graficar la media muestral obtenida como un punto
    plt.plot(medias_muestrales[i], i, 'o', color=color, markersize=3)

plt.title(f'Simulación de {n_corridas} Intervalos de Confianza al {nivel_confianza*100:.0f}%\n(Proporción de éxito observada: {proporcion_exito*100:.1f}%)', fontsize=14)
plt.xlabel('Valor de la Variable / Estimación', fontsize=12)
plt.ylabel('Número de Corrida (Simulación)', fontsize=12)
plt.grid(axis='x', linestyle=':', alpha=0.6)
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

## 3. Conclusión

Como se puede observar en la simulación y en el gráfico:
- Aproximadamente el **94%** (muy cercano al 95% teórico) de las 100 corridas logró capturar la media poblacional real (representada por la línea discontinua vertical).
- Esto demuestra de manera empírica que el término "95% de confianza" no aplica a un único intervalo en particular, sino que describe la **frecuencia a largo plazo** (la tasa de éxito) con la que el método de estimación captura el parámetro poblacional real si repitiéramos el muestreo infinitas veces.